# Ejemplo 1 · En clase: funciones de ventana con pandas
**Consigna:** para cada pedido mostrar (1) promedio del cliente, (2) diferencia contra ese promedio, (3) gasto acumulado del cliente, (4) monto de su compra anterior.

| SQL | pandas |
|---|---|
| `AVG(x) OVER(PARTITION BY c)` | `groupby(c)[x].transform('mean')` |
| `SUM(x) OVER(PARTITION BY c ORDER BY f)` | `groupby(c)[x].cumsum()` (tras ordenar) |
| `LAG(x)` | `groupby(c)[x].shift(1)` |

In [ ]:
import pandas as pd
pedidos = pd.DataFrame({
    "pedido_id": range(1, 9),
    "cliente_id": ["A","A","A","B","B","C","C","C"],
    "fecha": pd.to_datetime(["2024-01-05","2024-01-12","2024-01-20","2024-01-07",
                             "2024-01-15","2024-01-09","2024-01-18","2024-01-25"]),
    "monto": [100,200,150,500,300,80,120,400]})
pedidos

,pedido_id,cliente_id,fecha,monto
0,1,A,2024-01-05,100
1,2,A,2024-01-12,200
2,3,A,2024-01-20,150
3,4,B,2024-01-07,500
4,5,B,2024-01-15,300
5,6,C,2024-01-09,80
6,7,C,2024-01-18,120
7,8,C,2024-01-25,400


## Paso 1 · Ordenar (equivale al ORDER BY de la ventana)
Sin orden, `cumsum` y `shift` no tienen sentido.

In [ ]:
pedidos = pedidos.sort_values(['cliente_id','fecha']).reset_index(drop=True)
g = pedidos.groupby('cliente_id')['monto']

## Paso 2 · Promedio y diferencia (PARTITION BY)
`transform` devuelve una serie del mismo largo que el DataFrame: las filas **no se colapsan**, a diferencia de `agg`.

In [ ]:
pedidos['prom'] = g.transform('mean')
pedidos['dif'] = pedidos['monto'] - pedidos['prom']
pedidos

,pedido_id,cliente_id,fecha,monto,prom,dif
0,1,A,2024-01-05,100,150.0,-50.0
1,2,A,2024-01-12,200,150.0,50.0
2,3,A,2024-01-20,150,150.0,0.0
3,4,B,2024-01-07,500,400.0,100.0
4,5,B,2024-01-15,300,400.0,-100.0
5,6,C,2024-01-09,80,200.0,-120.0
6,7,C,2024-01-18,120,200.0,-80.0
7,8,C,2024-01-25,400,200.0,200.0


## Paso 3 · Acumulado y compra anterior (PARTITION BY + ORDER BY)

In [ ]:
pedidos['acum'] = g.cumsum()
pedidos['ant'] = g.shift(1)
pedidos

,pedido_id,cliente_id,fecha,monto,prom,dif,acum,ant
0,1,A,2024-01-05,100,150.0,-50.0,100,NaN
1,2,A,2024-01-12,200,150.0,50.0,300,100.0
2,3,A,2024-01-20,150,150.0,0.0,450,200.0
3,4,B,2024-01-07,500,400.0,100.0,500,NaN
4,5,B,2024-01-15,300,400.0,-100.0,800,500.0
5,6,C,2024-01-09,80,200.0,-120.0,80,NaN
6,7,C,2024-01-18,120,200.0,-80.0,200,80.0
7,8,C,2024-01-25,400,200.0,200.0,600,120.0


## Pregunta para discutir
¿Por qué la primera compra de cada cliente tiene `ant = NaN`? ¿Cómo la reemplazarías por 0? (`fillna(0)`, equivalente a `COALESCE`).

In [ ]:
pedidos['ant'] = pedidos['ant'].fillna(0)
pedidos

,pedido_id,cliente_id,fecha,monto,prom,dif,acum,ant
0,1,A,2024-01-05,100,150.0,-50.0,100,0.0
1,2,A,2024-01-12,200,150.0,50.0,300,100.0
2,3,A,2024-01-20,150,150.0,0.0,450,200.0
3,4,B,2024-01-07,500,400.0,100.0,500,0.0
4,5,B,2024-01-15,300,400.0,-100.0,800,500.0
5,6,C,2024-01-09,80,200.0,-120.0,80,0.0
6,7,C,2024-01-18,120,200.0,-80.0,200,80.0
7,8,C,2024-01-25,400,200.0,200.0,600,120.0
